In [7]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import warnings
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

# 忽略特定的 UserWarning 警告
warnings.filterwarnings("ignore", message="Glyph .* missing from current font.")

# 替换为实际数据集路径
labelled_data_path = 'kddcup_1_percent'
unlabelled_data_path = 'kddcup_1_percent.unlabeled'
except_path = 'kddcup_1_percent'

# 读取有标签的数据集
df_labelled = pd.read_csv(labelled_data_path, header=None)

# 读取无标签的数据集
df_unlabelled = pd.read_csv(unlabelled_data_path, header=None)
df_except = pd.read_csv(except_path, header=None)

# 定义列名
columns = ["duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", 
           "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", 
           "logged_in", "num_compromised", "root_shell", "su_attempted", 
           "num_root", "num_file_creations", "num_shells", "num_access_files", 
           "num_outbound_cmds", "is_host_login", "is_guest_login", "count", 
           "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", 
           "srv_rerror_rate", "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", 
           "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate", 
           "dst_host_diff_srv_rate", "dst_host_same_src_port_rate", 
           "dst_host_srv_diff_host_rate", "dst_host_serror_rate", 
           "dst_host_srv_serror_rate", "dst_host_rerror_rate", 
           "dst_host_srv_rerror_rate", "label"]

df_labelled.columns = columns
df_unlabelled.columns = columns[:-1]  # 无标签数据集没有label列
df_except.columns = columns

# 获取分类特征的所有种类
categorical_columns = ["protocol_type", "service", "flag"]

# 为所有分类特征创建一个LabelEncoder对象
le_dict = {col: LabelEncoder() for col in categorical_columns}

# Fit LabelEncoder对象于有标签和无标签数据集中
for col in categorical_columns:
    le = le_dict[col]
    le.fit(pd.concat([df_labelled[col], df_unlabelled[col], df_except[col]]))

# 对有标签数据集进行编码
for col in categorical_columns:
    le = le_dict[col]
    df_labelled[col] = le.transform(df_labelled[col])

# 对无标签数据集进行编码
for col in categorical_columns:
    le = le_dict[col]
    df_unlabelled[col] = le.transform(df_unlabelled[col])

# 对df_except进行编码
for col in categorical_columns:
    le = le_dict[col]
    df_except[col] = le.transform(df_except[col])

# 将标签编码
df_labelled['label'] = df_labelled['label'].apply(lambda x: 1 if x != 'normal.' else 0)
df_except['label'] = df_except['label'].apply(lambda x: 1 if x != 'normal.' else 0)

# 特征标准化
scaler = StandardScaler()

# 视角1特征基本主机基础
view1_features = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", 
    "land", "wrong_fragment", "urgent",
]

# 视角2特征 内容基础
view2_features = [
    "hot", "num_failed_logins", "logged_in", "num_compromised", "root_shell", 
    "su_attempted", "num_root", "num_file_creations", "num_shells", 
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login",
]

# 视角3 特征流量统计
view3_features = [
    "count", "srv_count", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
]

# 视角4特征其他
view4_features = [
    "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
]

# 特征标准化
df_labelled_view1 = df_labelled[view1_features].copy()
df_labelled_view2 = df_labelled[view2_features].copy()
df_labelled_view3 = df_labelled[view3_features].copy()
df_labelled_view4 = df_labelled[view4_features].copy()

df_labelled_view1[df_labelled_view1.columns] = scaler.fit_transform(df_labelled_view1[df_labelled_view1.columns])
df_labelled_view2[df_labelled_view2.columns] = scaler.fit_transform(df_labelled_view2[df_labelled_view2.columns])
df_labelled_view3[df_labelled_view3.columns] = scaler.fit_transform(df_labelled_view3[df_labelled_view3.columns])
df_labelled_view4[df_labelled_view4.columns] = scaler.fit_transform(df_labelled_view4[df_labelled_view4.columns])

# 划分有标签数据集的训练集和测试集
X_train_view1, X_test_view1, y_train, y_test = train_test_split(df_labelled_view1, df_labelled['label'], test_size=0.8, random_state=42)
X_train_view2, X_test_view2, _, _ = train_test_split(df_labelled_view2, df_labelled['label'], test_size=0.8, random_state=42)
X_train_view3, X_test_view3, _, _ = train_test_split(df_labelled_view3, df_labelled['label'], test_size=0.8, random_state=42)
X_train_view4, X_test_view4, _, _ = train_test_split(df_labelled_view4, df_labelled['label'], test_size=0.8, random_state=42)

# 无标签数据
X_train_unlabelled_view1 = df_unlabelled[view1_features].copy()
X_train_unlabelled_view2 = df_unlabelled[view2_features].copy()
X_train_unlabelled_view3 = df_unlabelled[view3_features].copy()
X_train_unlabelled_view4 = df_unlabelled[view4_features].copy()

X_train_unlabelled_view1[X_train_unlabelled_view1.columns] = scaler.fit_transform(X_train_unlabelled_view1[X_train_unlabelled_view1.columns])
X_train_unlabelled_view2[X_train_unlabelled_view2.columns] = scaler.fit_transform(X_train_unlabelled_view2[X_train_unlabelled_view2.columns])
X_train_unlabelled_view3[X_train_unlabelled_view3.columns] = scaler.fit_transform(X_train_unlabelled_view3[X_train_unlabelled_view3.columns])
X_train_unlabelled_view4[X_train_unlabelled_view4.columns] = scaler.fit_transform(X_train_unlabelled_view4[X_train_unlabelled_view4.columns])

# 显示数据集状态
print(f"Labelled Data Shape: {df_labelled.shape}")
print(f"Unlabelled Data Shape: {df_unlabelled.shape}")
print(f"Except Data Shape: {df_except.shape}")

# 显示分割后数据集的形状
print(f"X_train View1 Shape: {X_train_view1.shape}")
print(f"X_train View2 Shape: {X_train_view2.shape}")
print(f"X_train View3 Shape: {X_train_view3.shape}")
print(f"X_train View4 Shape: {X_train_view4.shape}")
print(f"y_train Shape: {y_train.shape}")
print(f"X_test View1 Shape: {X_test_view1.shape}")
print(f"X_test View2 Shape: {X_test_view2.shape}")
print(f"X_test View3 Shape: {X_test_view3.shape}")
print(f"X_test View4 Shape: {X_test_view4.shape}")
print(f"y_test Shape: {y_test.shape}")
print(f"X_train_unlabelled View1 Shape: {X_train_unlabelled_view1.shape}")
print(f"X_train_unlabelled View2 Shape: {X_train_unlabelled_view2.shape}")
print(f"X_train_unlabelled View3 Shape: {X_train_unlabelled_view3.shape}")
print(f"X_train_unlabelled View4 Shape: {X_train_unlabelled_view4.shape}")


Labelled Data Shape: (4940, 42)
Unlabelled Data Shape: (3111, 41)
Except Data Shape: (4940, 42)
X_train View1 Shape: (988, 9)
X_train View2 Shape: (988, 13)
X_train View3 Shape: (988, 10)
X_train View4 Shape: (988, 9)
y_train Shape: (988,)
X_test View1 Shape: (3952, 9)
X_test View2 Shape: (3952, 13)
X_test View3 Shape: (3952, 10)
X_test View4 Shape: (3952, 9)
y_test Shape: (3952,)
X_train_unlabelled View1 Shape: (3111, 9)
X_train_unlabelled View2 Shape: (3111, 13)
X_train_unlabelled View3 Shape: (3111, 10)
X_train_unlabelled View4 Shape: (3111, 9)


In [5]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import warnings

# 忽略特定的 UserWarning 警告
warnings.filterwarnings("ignore", message="Glyph .* missing from current font.")

# 初始化四个 RandomForestClassifier 模型，分别对应四个不同的视角
clf_view1 = RandomForestClassifier() 
clf_view2 = RandomForestClassifier()  
clf_view3 = RandomForestClassifier()  
clf_view4 = RandomForestClassifier()  

# 分别训练四个视角的模型
clf_view1.fit(X_train_view1, y_train)
clf_view2.fit(X_train_view2, y_train)
clf_view3.fit(X_train_view3, y_train)
clf_view4.fit(X_train_view4, y_train)

y_train_labelled1 = y_train.copy()
y_train_labelled2 = y_train.copy()
y_train_labelled3 = y_train.copy()
y_train_labelled4 = y_train.copy()

max_iterations = 10
confidence_threshold = 0.95

for iteration in range(max_iterations):
    # 检查所有无标签训练集是否为空，如果是，则终止循环
    if X_train_unlabelled_view1.empty and X_train_unlabelled_view2.empty and X_train_unlabelled_view3.empty and X_train_unlabelled_view4.empty:
        print(f"All unlabelled training sets are empty. Ending at iteration {iteration}.")
        break

    # 视角使用模型预测无标签数据的概率
    y_unlabelled_proba_view1 = clf_view1.predict_proba(X_train_unlabelled_view1)
    y_unlabelled_proba_view2 = clf_view2.predict_proba(X_train_unlabelled_view2)
    y_unlabelled_proba_view3 = clf_view3.predict_proba(X_train_unlabelled_view3)
    y_unlabelled_proba_view4 = clf_view4.predict_proba(X_train_unlabelled_view4)
    # 视角的预测的值
    y_unlabelled_view1 = clf_view1.predict(X_train_unlabelled_view1)
    y_unlabelled_view2 = clf_view2.predict(X_train_unlabelled_view2)
    y_unlabelled_view3 = clf_view3.predict(X_train_unlabelled_view3)
    y_unlabelled_view4 = clf_view4.predict(X_train_unlabelled_view4)

    # 计算四个视角的高置信度样本索引
    high_confidence_indices_view1 = np.where(np.max(y_unlabelled_proba_view1, axis=1) >= confidence_threshold)[0]
    high_confidence_indices_view2 = np.where(np.max(y_unlabelled_proba_view2, axis=1) >= confidence_threshold)[0]
    high_confidence_indices_view3 = np.where(np.max(y_unlabelled_proba_view3, axis=1) >= confidence_threshold)[0]
    high_confidence_indices_view4 = np.where(np.max(y_unlabelled_proba_view4, axis=1) >= confidence_threshold)[0]
    # 合并所有视角的高置信度样本索引
    union_indices_all = np.union1d(np.union1d(np.union1d(high_confidence_indices_view1, high_confidence_indices_view2), high_confidence_indices_view3), high_confidence_indices_view4)
    
    # 将高置信度样本及其标签合并到原始训练集中
    # 视角1
    X_train_view1 = pd.concat([X_train_view1, X_train_unlabelled_view1.iloc[high_confidence_indices_view2]])
    y_train_labelled1 = pd.concat([y_train_labelled1, pd.Series(y_unlabelled_view2[high_confidence_indices_view2])])

    X_train_view1 = pd.concat([X_train_view1, X_train_unlabelled_view1.iloc[high_confidence_indices_view3]])
    y_train_labelled1 = pd.concat([y_train_labelled1, pd.Series(y_unlabelled_view3[high_confidence_indices_view3])])

    X_train_view1 = pd.concat([X_train_view1, X_train_unlabelled_view1.iloc[high_confidence_indices_view4]])
    y_train_labelled1 = pd.concat([y_train_labelled1, pd.Series(y_unlabelled_view4[high_confidence_indices_view4])])
    
    # 视角2
    X_train_view2 = pd.concat([X_train_view2, X_train_unlabelled_view2.iloc[high_confidence_indices_view1]])
    y_train_labelled2 = pd.concat([y_train_labelled2, pd.Series(y_unlabelled_view1[high_confidence_indices_view1])])

    X_train_view2 = pd.concat([X_train_view2, X_train_unlabelled_view2.iloc[high_confidence_indices_view3]])
    y_train_labelled2 = pd.concat([y_train_labelled2, pd.Series(y_unlabelled_view3[high_confidence_indices_view3])])

    X_train_view2 = pd.concat([X_train_view2, X_train_unlabelled_view2.iloc[high_confidence_indices_view4]])
    y_train_labelled2 = pd.concat([y_train_labelled2, pd.Series(y_unlabelled_view4[high_confidence_indices_view4])])

    # 视角3
    X_train_view3 = pd.concat([X_train_view3, X_train_unlabelled_view3.iloc[high_confidence_indices_view1]])
    y_train_labelled3 = pd.concat([y_train_labelled3, pd.Series(y_unlabelled_view1[high_confidence_indices_view1])])

    X_train_view3 = pd.concat([X_train_view3, X_train_unlabelled_view3.iloc[high_confidence_indices_view2]])
    y_train_labelled3 = pd.concat([y_train_labelled3, pd.Series(y_unlabelled_view2[high_confidence_indices_view2])])

    X_train_view3 = pd.concat([X_train_view3, X_train_unlabelled_view3.iloc[high_confidence_indices_view4]])
    y_train_labelled3 = pd.concat([y_train_labelled3, pd.Series(y_unlabelled_view4[high_confidence_indices_view4])])

    # 视角4
    X_train_view4 = pd.concat([X_train_view4, X_train_unlabelled_view4.iloc[high_confidence_indices_view1]])
    y_train_labelled4 = pd.concat([y_train_labelled4, pd.Series(y_unlabelled_view1[high_confidence_indices_view1])])

    X_train_view4 = pd.concat([X_train_view4, X_train_unlabelled_view4.iloc[high_confidence_indices_view2]])
    y_train_labelled4 = pd.concat([y_train_labelled4, pd.Series(y_unlabelled_view2[high_confidence_indices_view2])])

    X_train_view4 = pd.concat([X_train_view4, X_train_unlabelled_view4.iloc[high_confidence_indices_view3]])
    y_train_labelled4 = pd.concat([y_train_labelled4, pd.Series(y_unlabelled_view3[high_confidence_indices_view3])])

    # 从未标记的数据集中删除已选择的高置信度样本
    X_train_unlabelled_view1 = X_train_unlabelled_view1.drop(X_train_unlabelled_view1.index[union_indices_all])
    X_train_unlabelled_view2 = X_train_unlabelled_view2.drop(X_train_unlabelled_view2.index[union_indices_all])
    X_train_unlabelled_view3 = X_train_unlabelled_view3.drop(X_train_unlabelled_view3.index[union_indices_all])
    X_train_unlabelled_view4 = X_train_unlabelled_view4.drop(X_train_unlabelled_view4.index[union_indices_all])

    # 重新训练四个视角的模型
    clf_view1.fit(X_train_view1, y_train_labelled1)
    clf_view2.fit(X_train_view2, y_train_labelled2)
    clf_view3.fit(X_train_view3, y_train_labelled3)
    clf_view4.fit(X_train_view4, y_train_labelled4)

    print(f"Iteration {iteration + 1}: Added {len(union_indices_all)} samples to the training set.")

print(len(X_train_unlabelled_view4))
# 模型评估
y_pred_view1 = clf_view1.predict(X_test_view1)
y_pred_view2 = clf_view2.predict(X_test_view2)
y_pred_view3 = clf_view3.predict(X_test_view3)
y_pred_view4 = clf_view4.predict(X_test_view4)

# 计算每个样本的预测概率的平均值
y_proba_avg = (clf_view1.predict_proba(X_test_view1) + clf_view2.predict_proba(X_test_view2) + clf_view3.predict_proba(X_test_view3) + clf_view4.predict_proba(X_test_view4)) / 4

# 基于平均概率选择最终预测类别
y_pred_combined = np.argmax(y_proba_avg, axis=1)

# 输出最终的预测准确率和分类报告
print("Combined Model Accuracy:", accuracy_score(y_test, y_pred_combined))
print("Combined Model Classification Report:")
print(classification_report(y_test, y_pred_combined))

Iteration 1: Added 2816 samples to the training set.
Iteration 2: Added 291 samples to the training set.
Iteration 3: Added 3 samples to the training set.
Iteration 4: Added 0 samples to the training set.
Iteration 5: Added 0 samples to the training set.
Iteration 6: Added 0 samples to the training set.
Iteration 7: Added 0 samples to the training set.
Iteration 8: Added 0 samples to the training set.
Iteration 9: Added 0 samples to the training set.
Iteration 10: Added 0 samples to the training set.
1
Combined Model Accuracy: 0.9896255060728745
Combined Model Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.95      0.97       775
           1       0.99      1.00      0.99      3177

    accuracy                           0.99      3952
   macro avg       0.99      0.98      0.98      3952
weighted avg       0.99      0.99      0.99      3952



In [8]:
# import pandas as pd
# from sklearn.preprocessing import LabelEncoder, StandardScaler
# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, classification_report
# import numpy as np
# import warnings


# # 忽略特定的 UserWarning 警告
# warnings.filterwarnings("ignore", message="Glyph .* missing from current font.")

# # 初始化四个随机森林模型，分别对应四个不同的视角
# clf_view1 = RandomForestClassifier(n_estimators=100)
# clf_view2 = RandomForestClassifier(n_estimators=100)
# clf_view3 = RandomForestClassifier(n_estimators=100)
# clf_view4 = RandomForestClassifier(n_estimators=100)

# # 分别训练四个视角的模型
# clf_view1.fit(X_train_view1, y_train)
# clf_view2.fit(X_train_view2, y_train)
# clf_view3.fit(X_train_view3, y_train)
# clf_view4.fit(X_train_view4, y_train)

# max_iterations = 10
# confidence_threshold = 0.8

# for iteration in range(max_iterations):
#     # 视角1使用模型预测无标签数据的概率
#     y_unlabelled_proba_view1 = clf_view1.predict_proba(X_train_unlabelled_view1)
#     # 视角2使用模型预测无标签数据的概率
#     y_unlabelled_proba_view2 = clf_view2.predict_proba(X_train_unlabelled_view2)
#     # 视角3使用模型预测无标签数据的概率
#     y_unlabelled_proba_view3 = clf_view3.predict_proba(X_train_unlabelled_view3)
#     # 视角4使用模型预测无标签数据的概率
#     y_unlabelled_proba_view4 = clf_view4.predict_proba(X_train_unlabelled_view4)
    
#     # 计算四个视角的高置信度样本索引的交集
#     high_confidence_indices_view1 = np.where(np.max(y_unlabelled_proba_view1, axis=1) >= confidence_threshold)[0]
#     high_confidence_indices_view2 = np.where(np.max(y_unlabelled_proba_view2, axis=1) >= confidence_threshold)[0]
#     high_confidence_indices_view3 = np.where(np.max(y_unlabelled_proba_view3, axis=1) >= confidence_threshold)[0]
#     high_confidence_indices_view4 = np.where(np.max(y_unlabelled_proba_view4, axis=1) >= confidence_threshold)[0]

#     # 使用np.intersect1d找到交集
#     high_confidence_indices_inter = np.intersect1d(high_confidence_indices_view1, high_confidence_indices_view2)
#     high_confidence_indices_inter = np.intersect1d(high_confidence_indices_inter, high_confidence_indices_view3)
#     high_confidence_indices_inter = np.intersect1d(high_confidence_indices_inter, high_confidence_indices_view4)

#     # 从交集中选择高置信度样本及其预测标签
#     X_high_confidence_view1_inter = X_train_unlabelled_view1.iloc[high_confidence_indices_inter]
#     y_high_confidence_view1_inter = np.argmax(y_unlabelled_proba_view1[high_confidence_indices_inter], axis=1)

#     X_high_confidence_view2_inter = X_train_unlabelled_view2.iloc[high_confidence_indices_inter]
#     y_high_confidence_view2_inter = np.argmax(y_unlabelled_proba_view2[high_confidence_indices_inter], axis=1)

#     X_high_confidence_view3_inter = X_train_unlabelled_view3.iloc[high_confidence_indices_inter]
#     y_high_confidence_view3_inter = np.argmax(y_unlabelled_proba_view3[high_confidence_indices_inter], axis=1)

#     X_high_confidence_view4_inter = X_train_unlabelled_view4.iloc[high_confidence_indices_inter]
#     y_high_confidence_view4_inter = np.argmax(y_unlabelled_proba_view4[high_confidence_indices_inter], axis=1)
    
#     # 将高置信度样本及其标签合并到原始训练集中
#     X_train_view1_combined = pd.concat([X_train_view1, X_high_confidence_view1_inter])
#     y_train_view1_combined = np.concatenate([y_train, y_high_confidence_view1_inter])

#     X_train_view2_combined = pd.concat([X_train_view2, X_high_confidence_view2_inter])
#     y_train_view2_combined = np.concatenate([y_train, y_high_confidence_view2_inter])

#     X_train_view3_combined = pd.concat([X_train_view3, X_high_confidence_view3_inter])
#     y_train_view3_combined = np.concatenate([y_train, y_high_confidence_view3_inter])

#     X_train_view4_combined = pd.concat([X_train_view4, X_high_confidence_view4_inter])
#     y_train_view4_combined = np.concatenate([y_train, y_high_confidence_view4_inter])

#     # 从未标记的数据集中删除已选择的高置信度样本
#     X_train_unlabelled_view1 = X_train_unlabelled_view1.drop(X_high_confidence_view1_inter.index)
#     X_train_unlabelled_view2 = X_train_unlabelled_view2.drop(X_high_confidence_view2_inter.index)
#     X_train_unlabelled_view3 = X_train_unlabelled_view3.drop(X_high_confidence_view3_inter.index)
#     X_train_unlabelled_view4 = X_train_unlabelled_view4.drop(X_high_confidence_view4_inter.index)

#     # 重新训练四个视角的模型
#     clf_view1.fit(X_train_view1_combined, y_train_view1_combined)
#     clf_view2.fit(X_train_view2_combined, y_train_view2_combined)
#     clf_view3.fit(X_train_view3_combined, y_train_view3_combined)
#     clf_view4.fit(X_train_view4_combined, y_train_view4_combined)

#     print(f"Iteration {iteration + 1}: Added {len(high_confidence_indices_inter)} samples to the training set.")

# # 模型评估
# y_pred_view1 = clf_view1.predict(X_test_view1)
# y_pred_view2 = clf_view2.predict(X_test_view2)
# y_pred_view3 = clf_view3.predict(X_test_view3)
# y_pred_view4 = clf_view4.predict(X_test_view4)

# # 计算每个样本的预测概率的平均值
# y_proba_avg = (clf_view1.predict_proba(X_test_view1) + clf_view2.predict_proba(X_test_view2) + clf_view3.predict_proba(X_test_view3) + clf_view4.predict_proba(X_test_view4)) / 4

# # 基于平均概率选择最终预测类别
# y_pred_combined = np.argmax(y_proba_avg, axis=1)

# # 输出最终的预测准确率和分类报告
# print("Combined Model Accuracy:", accuracy_score(y_test, y_pred_combined))
# print("Combined Model Classification Report:")
# print(classification_report(y_test, y_pred_combined))

Iteration 1: Added 509 samples to the training set.
Iteration 2: Added 1911 samples to the training set.
Iteration 3: Added 320 samples to the training set.
Iteration 4: Added 14 samples to the training set.
Iteration 5: Added 243 samples to the training set.
Iteration 6: Added 14 samples to the training set.
Iteration 7: Added 9 samples to the training set.
Iteration 8: Added 8 samples to the training set.
Iteration 9: Added 1 samples to the training set.
Iteration 10: Added 0 samples to the training set.
Combined Model Accuracy: 0.9906376518218624
Combined Model Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.98       775
           1       0.99      1.00      0.99      3177

    accuracy                           0.99      3952
   macro avg       0.99      0.98      0.98      3952
weighted avg       0.99      0.99      0.99      3952

